name: 241211_data_analysis \
date: 02/04/2025 \
version: 2.2 \
github root: #google colab stream: version 2.0\
author: Justin Sankey, Johanna Ganglbauer

**description**: Takes raw liquit chromatography mass spectroscopy (LCMS) data (exported table from SCIEX Analyst Software), converts mass (ng/sample) to concentration, evaluates QAQC criteria: (i) retention time shifts, (ii) method detection limits, (iii) recovery rates, and (iv) ion abundance ratios.
Generates plots, writes results to excel, and creates long format table (.csv) for data publications.

**changes in comparison to previous version**:
*   'degenerate' in concentration declared as nan
*   removed 0.5 mL per sample from inputs
*   included final concentration table with numeric values only
*   sorted IPS in graph accordingly
*   included quantification from TOF channel as output (for now only concentrations, no recovery, mdl etc.)
*   replaced mass channel ratios with deviation of ion abundance ratios
*   within the MDL calculation: concentration < IDL are replaced with IDL - ensures MDL is never smaller than IDL, avoids high standard deviations for multiple concentrations below IDL
*   optional outlier detection for MDL calculation
*   concentration boxplots for MDL calculation
*   included retention time quality control in final table and long format table: calculated average shift between IS and compound in calibration and compared to shifts in samples for all the compounds
*   simplified final output and output tables with only four QAQC criteria: \
(i) retention time difference (RTD), (ii) below detection limit (BDL), (iii) poor recovery rate (Poor RR),
(iv) poor ion abundance ratio deviation (Poor IARD)
*   cleaned up 'old' recovery parts
*   major restructuring: run create_project_folder.ipynb before running data analysis. Folder structure strictly given -> less effort to setup project
*   major cleanup: put functions into utils.py for better readability
*   major restructuing: all simulation parameters are csv input now
*   drop data where "Used" column is set to False
*   output final concentration values with 4 valid digits (not 4 digits after komma, but for digits in total, so either 1011, 101.1, 10.11, 1.011 or 0.1001)
*   for RT difference display difference from reference midpoint compound, when chemical identical internal standard is not available.
*   for MDL: include calculation for all TOF channels, beautify plots


**what needs to be implemented**:
*   for IPS area plot: add explanation, replace quantification with 'samples', doublecheck if normalization works
*   remove response factor and ion abundance ratios in calibration from excel
*   JB: read in different IDL files depending on sample matrix
*   JG: include EPA thresholds for recoveries
*   JG: further calculations with TOF MS channel (?)
*   SV: extract QAQC samples from researcher data and collect them somewhere (?)
*   JB: once enough data has been collected calculate 'default MDLs'
*   SV, JB: eventually include CCV in QAQC
*   **feel free to add your thoughts!**

**contact/help/complaints:** johanna.ganglbauer@uri.edu

# Loading packages, Settings
The block below will load all python packages needed for the following analysis.\
Moreover, it sets the display options to show data tables in your console and suppresses warnings

In [ ]:
# import all needed packages
import os
import numpy as np
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import pandas as pd

# import functions from utils.py
from utils import (
    read_in_data_files, get_sample_id_and_name, get_compounds_and_standards,
    parse_project_folder_structure, clean_up_data, round_to_n_sigfigs)

# import and suppress warnings
import warnings
warnings.filterwarnings('ignore')

# display settings
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None)  # Show full width of columns

# Change project_folder variable in current code block
Make sure you setup the project folder by following the instructions in the notebook **create_project_folder.ipynb** before you run the current notebook.
Change the variable **project_folder** in the first Code block of the notebook, so that it points to the project folder you created with the notebook **create_project_folder.ipynb**.
You can copy the path of your project folder on windows by right clicking on the folder in your file explorer and choosing the option *Copy as Path*. \
The same works on Mac OS: Control-click or right-click on the folder in Finder. Press the Option (Alt) key. Choose *Copy [foldername] as Pathname*.

In [ ]:
### INPUT FILEPATHS
project_folder = 'test'

# The following tests if the project folder has the desired structure and throws errors if not.
parse_project_folder_structure(project_folder)

# EXPERT AREA for Inputs
Usually you can leave the Code block below unchanged.\
Provides advanced simulation paramters used by the Code. 

In [ ]:
# file paths for IDL and IQL data - not meant to be adopted
idl_filepath = r'lab_parameters/IDL_2024.csv'

# idl default value
idl_unknown = 1e-3

# used to identify internal standards from column "Component Name" - usually you do not have to change it
standard_identifiers = 'EIS|NIS|IDA|IPS|13C|d-|d3-|d5-|18O'

# Set to True if outliers should not be considered for MDL calculation. Outliers are shown in the boxplot respectively. -> Remove in next version
exclude_outlier_for_mdl = False

# limits for concentration values in plots
concentration_limit_plot = 2

# limits for recovery rate values in plots
recovery_limit_plot = 5e2

### OUTPUT FILLEPATHS
# processed data output excel/csv file path - without .xlsx/.csv ending
processed_filepath_csv = os.path.join(project_folder, 'processed_data', 'output.csv')
processed_filepath_xlsx = os.path.join(project_folder, 'processed_data', 'output.xlsx')

# directory to save plots to
plot_directory = os.path.join(project_folder, 'processed_data', 'plots')

# Read in simulation parameters
(i) simulation parameters from simulation_parameters.csv \
(ii) sample_parameters from sample_parameters.csv. In addition, the predominant unit used within the data is extracted and the average sample size. Both will be used for the normalization of MDLs at a later stage\
(iii) recovery thresholds from recovery_thresholds.csv \

In [ ]:
# read csv file as data frame
simulation_parameters = pd.read_csv(os.path.join(project_folder, 'code_parameters', 'simulation_parameters.csv'), index_col=[0])

# extract all simulation parameters from csv file, parse them and save them to variables used in the codeblocks to follow.
eis_identifier = simulation_parameters.loc['EIS identifier', 'Parameter Value']
nis_identifier = simulation_parameters.loc['NIS identifier', 'Parameter Value']
calibration_midpoint_identifier = simulation_parameters.loc['calibration midpoint identifier', 'Parameter Value']
channel_selection = simulation_parameters.loc['channel selection', 'Parameter Value']
# sanity check - will produce an error if you indicated a variable recovery_data which is not allowed.
if channel_selection not in ['core', 'extended', 'average']:
    raise NameError("""
        Invalid input for variable channel_selection in your simulation_parameters.csv, use either 'core', 'extended', or 'average'.
                    """)
iard_threshold = float(simulation_parameters.loc['IAR deviation threshold', 'Parameter Value'])
rt_threshold = float(simulation_parameters.loc['retention time difference threshold', 'Parameter Value'])

In [ ]:
# read in sample input data
sample_input_data = pd.read_csv(os.path.join(project_folder, 'code_parameters', 'sample_parameters.csv'), index_col=[0])

# save relation of index and sample name in publication
sample_number_mapper_sample_name = sample_input_data['alternative name (used in results)'].to_dict()
# save relation of index and unit in publication
sample_number_mapper_sample_unit = sample_input_data['unit (e.g. g/mL/sample)'].to_dict()

# get and save dominant unit
sample_dominant_unit = sample_input_data.loc[~sample_input_data['used for mdl calculation'],'unit (e.g. g/mL/sample)'].value_counts().nlargest(1).index[0]
# get and save average sample size
sample_average_size = sample_input_data.loc[~sample_input_data['used for mdl calculation'],'volume/weight/number of samples'].mean()

In [ ]:
# read in recovery threshold limits and save them as dictionary
recovery_thresholds = pd.read_csv(os.path.join(project_folder, 'code_parameters', 'recovery_thresholds.csv'), index_col=[0])
recovery_thresholds_dict = recovery_thresholds.to_dict()

# Read in data and clean up 
The following Code block reads in all raw data files and cleans up data. Uses functions read_in_data_files(), get_sample_id_and_name(), and clean_up_data() from utils.py.\
Makes sure the component IPS-13C2_PFOA is not used as non extracted standard, replaces it with IPS-13C4_PFOA - relevant for 'older data' originating from 2024, hopefully obsolete soon.

In [ ]:
# calls function to read in data
data = read_in_data_files(project_folder=project_folder)

# extract sample names and compound names from raw data
sample_list = get_sample_id_and_name(data)

# calls function to get complete list of samples
data = clean_up_data(data=data, sample_list=sample_list, channel_selection=channel_selection)

# Delete all '13C2_PFOA_TOF MS' channel rows if they are in the data set
data = data[~data['Component Name'].isin(['13C2_PFOA_TOF MS'])]

# Replace 'IPS-13C2_PFOA' values: optional, only when component name occurs
if any(data['Component Name'].isin(['IPS-13C4_PFOA'])):
    data['Component Group Name'] = data['Component Group Name'].replace('IPS-13C2_PFOA', 'IPS-13C4_PFOA')

    # Find rows where 'Component Group Name' is 'IPS-13C4_PFOA' (after replacement)
    mask = data['Component Group Name'] == 'IPS-13C4_PFOA'

    # Iterate through each of these rows and replace area in column
    for idx, row in data[mask].iterrows():
        sample_name = row['Sample Name']

        # Find the corresponding row with 'Component Name' == 'IPS-13C4_PFOA' and the same 'Sample Name'
        matching_row = data[(data['Component Name'] == 'IPS-13C4_PFOA') & (data['Sample Name'] == sample_name)]

        if not matching_row.empty:
            # Update the 'Area IPS' with the value from 'Area' in the matching row
            data.at[idx, 'Area IPS'] = matching_row['Area'].values[0]

# Get compound names from raw data
Calls get_compounds_and_standards() from utils.py. Saves:
- eis_sorted: list of mass labeled extracted internal standards (EIS) with the compound order preserved from the order in the raw data
- nis_sorted: list of mass labeled non-extracted internal standards (NIS) with the compound order preserved from the order in the raw data
- pfas_compounds_msms: list of pfas target analytes from the MS/MS channel, with the compound order preserved from the order in the raw data
- pfas_compounds_tof: list of pfas target analytes from the TOF channel, with the compound order preserved from the order in the raw data

In [ ]:
# extract list of compound names and standard names from raw data
compounds_msms, compounds_tof, eis_nis_msms, eis_nis_tof = get_compounds_and_standards(
    data=data, sample_list=sample_list, standard_identifiers=standard_identifiers
    )

# get list of sorted IDA and IPS names from msms channel
eis_sorted = [standard for standard in eis_nis_msms if (not pd.isnull(standard) and eis_identifier in standard)]
nis_sorted = [standard for standard in eis_nis_msms if (not pd.isnull(standard) and nis_identifier in standard)]

# get rid of np.nan component names when used as index to create tables
pfas_compounds_msms = [component for component in compounds_msms if not pd.isnull(component)]
pfas_compounds_tof = [component for component in compounds_tof if not pd.isnull(component)]

# Seperate data in calibration, calibration midpoint and samples
The following code block separates data in calibration data, calibration midpoint data and sample data, ccv_data, and ccv_midpoint_data.

In [ ]:
# Split data into quantification data, calibration data, and blanks for mdl calculation
calibration_data = data[(data['Sample Type'] == 'Standard')]
calibration_midpoint_data = calibration_data[(calibration_data['Sample ID'].str.contains(calibration_midpoint_identifier))]
sample_data = data[(data['Sample Type'] != 'Standard')]
ccv_data = sample_data[(sample_data['Sample ID'].str.contains(f'CS0|{calibration_midpoint_identifier}'))]
ccv_midpoint_data = ccv_data[(ccv_data['Sample ID'].str.contains(calibration_midpoint_identifier))]

# Create tables for detected pfas masses
Pivot tables are created to construct and export tables containing the 'Calculated Concentration' information from the raw data.
The 'Calculated Concentration' information corresponds to ng of PFAS detected in the sample, respectively.

Tables for both channels MS/MS and TOF are exported to excel.

In [ ]:
pfas_mass = sample_data.pivot_table(
    index=('Sample Index'), columns='Component Name', values='Calculated Concentration', dropna=False,
)
pfas_mass.rename(index=sample_number_mapper_sample_name , inplace=True)
pfas_mass_msms = pfas_mass[pfas_compounds_msms]
pfas_mass_tof = pfas_mass[pfas_compounds_tof]

# write initial concentrations to excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl') as writer:
    pfas_mass_msms.to_excel(writer, sheet_name='MSMS Mass (ng per Sample)')
    pfas_mass_tof.to_excel(writer, sheet_name='TOF Mass (ng per Sample)')

# Visual Sanity Check 1:
The IPS Areas are compared in the calibration data and the samples themselves. The areas of the calibration data are internally multiplied by two in order two account for the differences in volume (calibration: 1 mL, quantification: 0.5 mL).

In most of the cases you add 4 ng internal standard to your 0.5 mL samples: the areas of the corresponding IPS should be in a comparable range for calibration data (left) and your samples (right) in the boxplots below.

**Doublecheck that the information you provided in the "Actual Concentration" column considering the amount of IPS and IDA you added to your samples is correct! This will affect your recovery rates. Usually you add 4 ng/(0.5 mL sample). Check the title of the right boxplot**.

In the following code there are no QAQC criteria, which flag samples with highly deviating IPS areas, it is your responsibility to check the ranges. It will affect the recovery rates, so it is a good idea to make sure that the areas are in comparable ranges, or you can explain why they are not.

In [ ]:
# Calculate IPS average area per compound in calibration data and plot it
calibration_only_ips = calibration_data[calibration_data['Component Name'].str.contains('IPS')]
calibration_only_ips['Area'] = calibration_only_ips['Area']
quantification_blank_only_ips = sample_data[sample_data['Component Name'].str.contains('IPS')]

# evaluate mean per component
calibration_ips_area_averages = calibration_only_ips.groupby('Component Name')['Area'].mean()
quantification_blank_ips_area_averages = quantification_blank_only_ips.groupby('Component Name')['Area'].mean()

# evaluate IPS concentration in quantification
ips_concentration = quantification_blank_only_ips['Actual Concentration'].mean()

# create plot for comparison
image_path = os.path.join(plot_directory, 'ips_areas.png')
fig, ax = plt.subplots(ncols=2, figsize=(8, 8), sharey=True)
calibration_only_ips_grouped = calibration_only_ips.groupby('Component Name', sort=False)
calibration_only_ips_grouped.boxplot(column='Area', ax=ax[0], subplots=False)
ax[0].plot([np.nan] + calibration_ips_area_averages[nis_sorted].to_list(), color='red', linestyle='', marker="o", label='calibration average')
quantification_blank_only_ips_grouped = quantification_blank_only_ips.groupby('Component Name', sort=False)
quantification_blank_only_ips_grouped.boxplot(column='Area', ax=ax[1], subplots=False)
ax[1].plot([np.nan] + quantification_blank_ips_area_averages[nis_sorted].to_list(), color='red', linestyle='', marker="o", label='quantification average')
fig.suptitle('')
ax[0].set_title('Calibration: 4 ng/mL')
ax[1].set_title(f'Samples: {round(ips_concentration, 0)} ng/(0.5 mL)?')
ax[0].set_xticks(
    ticks=range(len(calibration_ips_area_averages) + 1),
    labels=[''] + nis_sorted, rotation=90
    )
ax[1].set_xticks(
    ticks=range(len(quantification_blank_ips_area_averages) + 1),
    labels=[''] + nis_sorted, rotation=90
    )
ax[0].set_ylabel('IPS area')
[this_ax.set_xlabel('') for this_ax in ax]
plt.savefig(image_path, bbox_inches='tight')
plt.show()

# save IPS area comparison plot to excel file
workbook = load_workbook(processed_filepath_xlsx)
plot_sheet = workbook.create_sheet('MSMS IPS Area')

img = Image(os.path.join(plot_directory, 'ips_areas.png'))

cell_position = plot_sheet.cell(row=1, column=1).coordinate
plot_sheet.add_image(img, cell_position)

workbook.save(processed_filepath_xlsx)

In the following code block the data (calibration not included) is separated in the data junk used for mdl calculation,
and the remaining data (where the PFAS are quantified).

In [ ]:
indices_for_mdl_evaluation = sample_input_data[sample_input_data['used for mdl calculation']].index

if len(indices_for_mdl_evaluation) == 0:
    mdl_only = None
    quantification_only = sample_data
    print(f'Be careful, no samples have been collected for the MDL calculation.')
else:
    mdl_only = sample_data[sample_data['Sample Index'].isin(indices_for_mdl_evaluation)]
    quantification_only = sample_data[~sample_data['Sample Index'].isin(indices_for_mdl_evaluation)]

# QAQC Criterion 1: Retention Time Shift (RTS)
In each sample compare retention time of PFAS compound with retention time of corresponding extracted internal standard (EIS).\
For Compounds with the molecular structure of the mass labeled internal standard being different than the molecular structure of the compound itself, the average difference of the retention time of the related extracted internal standard is calculated from calibration midpoint samples.

If available the calibration midpoint from continous calibration verification (CCV) is available, it is used as reference, otherwise the calibration midpoint from the calibration data itself is considered.

The first Code Block below evaluates average retention time differences between pfas target compound and corresponding internal standard from the selected calibration midpoint. Is set to zero if chemical structure of analytical standard and corresponding internal standard is the same. Differences (or deltas) are saved to excel sheet 'RT shift'.

Second Code Block below provides functions to 'highlight' cells in excel red, based on boolean mask, as well as a function to rename samples (rows) in excel tables.

Third Code Block below calculates retention time differences (or deltas) between analytical standard and corresponding internal standard and compares to threshold rt_threshold, which is defined in the simulation parameters.

In [ ]:
# evaluate retention time of reference (=mid point from continous calibration verification or calibration)

# check if ccv_midpoint data is available and use it as reference point, otherwise choose calibration midpoint from calibration data as reference.
if not ccv_midpoint_data.empty:
    # extract relavant columns from calibration data
    mid_point = ccv_midpoint_data[['Sample Name', 'Component Name', 'Retention Time', 'IS Name']]
else:
    print('Calibration midpoint from continuos calibration verification is not available, reference chosen from calibration data.')
    mid_point = calibration_midpoint_data[['Sample Name', 'Component Name', 'Retention Time', 'IS Name']]

# calculate mean retention time from selected reference point
mid_point_rt = mid_point.groupby('Component Name')['Retention Time'].mean()

# initial dataframe which saves acceptable retention time shifts from selected reference point 
rt_shift_df = pd.DataFrame({"RT":[], "IS RT":[], 'RT diff':[]})

# loop over all compounds of selected reference midpoint
for index, retention_time in mid_point_rt.items():
    # skip internal standards
    if any(substring in index for substring in [eis_identifier, nis_identifier] + standard_identifiers.split('|')):
        continue
    # get name of corresponding internal standard (IDA)
    is_name = mid_point.loc[mid_point['Component Name']==index, 'IS Name'].to_list()[0]
    # get retention time of corresponding inernal standard (IDA) in calibration mid point
    is_rt = mid_point_rt[is_name]
    # if molecular structure (and naming) of compound and corresponding internal standard 
    # are the same, no additional delta in retention time is allowed
    if index in is_name:
        rt_shift_df.loc[index] = [retention_time, is_rt, 0]
    # if molecular structure (and naming) of compound and corresponding internal standard
    # are not the same, save addition delta in retention time from calibration midpoint
    else:
        rt_shift_df.loc[index] = [retention_time, is_rt, round_to_n_sigfigs(x=retention_time - is_rt,n=4)]
        # use retention time of target compound in the calibration midpoint as reference and save it to the column 'IS Retention Time'
        sample_data.loc[sample_data['Component Name']==index, 'IS Retention Time'] = retention_time

# extract tof data and put TOF information from compound name to column header
rt_shift_tof = rt_shift_df.loc[pfas_compounds_tof]  # extract in right chemical order
rt_shift_tof.index = [item[0] for item in rt_shift_tof.index.str.split('_')]  # reindex with msms names
rt_shift_tof.columns = [elem + ' TOF' for elem in rt_shift_tof.columns]  # add 'TOF' to column names

rt_shift_msms = rt_shift_df.loc[pfas_compounds_msms]
rt_shift_msms.columns = [elem + ' MS/MS' for elem in rt_shift_msms.columns]  # add 'MS/MS' to column names

# merge MSMS data with TOF data and save it to data frame rt_shift_excel
rt_shift_excel = pd.concat([rt_shift_msms, rt_shift_tof], axis=1)

# write sample input data and retention time shifts to excel
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    rt_shift_excel.to_excel(writer, sheet_name='RT Differences')

In [ ]:
# function to color excel according to threshold values / mask
def highlight(data, flag):
    '''Sets all data elements to red background, when flag is True. '''
    if data.ndim == 1:  # Series from .apply(axis=0) or axis=1
        return ['background-color: red' if v else '' for v in flag]
    else:  # from .apply(axis=None)
        return pd.DataFrame(np.where(flag, 'background-color: red', ''),
                            index=data.index, columns=data.columns)

# function to replace sample names in excel sheet accordingly
def reindex_in_excel(
        filename: str, sheetname: str, index_mapper: dict=sample_number_mapper_sample_name,
        ) -> None:
    """Function to rename index columns in excel."""
    # load excel file
    workbook = load_workbook(filename=filename)
    # open workbook
    sheet = workbook.active
    # get right sheet
    sheet = workbook[sheetname]
    # read in index column
    sample_index = [column.value for column in sheet['A']]
  
    # change index column values
    if sample_index[0] == 'Sample Index':
        for (row_index, sample_index) in enumerate(sample_index[1:]):
            sheet.cell(row=row_index + 1, column=1).value = index_mapper[sample_index]
    
    #save the file
    workbook.save(filename=filename)
    workbook.close()

In [ ]:
# initialize new retention_time_data frame and select columns
selected_columns = [
    'Sample Name', 'Sample Index', 'Component Name', 'Retention Time', 'IS Retention Time'
]
retention_time_data = sample_data[selected_columns]

# select only channel names of precursor masses (confirmation) and fragmented masses (quantification)
retention_time_data = retention_time_data[retention_time_data['Component Name'].isin(compounds_msms + compounds_tof)]

# calculate deviation from retention time of PFAS component and related internal injection standard (IDA)
retention_time_data['Delta Retention Time IS'] = \
    (retention_time_data['Retention Time'] - retention_time_data['IS Retention Time']).apply(lambda x: round_to_n_sigfigs(x=x, n=4))

# initialize new column RTD indicating the limits
retention_time_data['RTD'] = False
retention_time_data.loc[(
    (retention_time_data['Delta Retention Time IS'] < -rt_threshold) | (retention_time_data['Delta Retention Time IS'] > rt_threshold)
    ), 'RTD'] = True

nan_flag = retention_time_data['Delta Retention Time IS'].isnull()  # save mask of NaN values
retention_time_data.loc[nan_flag, 'RTD'] = np.nan  # set mask values of nan retention time values to np.nan

for (compounds, sheet_name) in zip([pfas_compounds_msms, pfas_compounds_tof], ['MSMS Delta RT in min', 'TOF Delta RT in min']):
    # Put retention time data in pivot tables
    retention_time_is = retention_time_data.pivot_table(
        index=('Sample Index',), columns='Component Name', values='Delta Retention Time IS', dropna=False,
    )[compounds]
    retention_time_is_flag = retention_time_data.pivot_table(
        index=('Sample Index',), columns='Component Name', values='RTD', dropna=False,
    )[compounds]
    display(retention_time_is)

    # color cells based on threshold values
    retention_time_is_styled = retention_time_is.style.apply(
        highlight, flag=retention_time_is_flag.fillna(value=False), axis=None
        )

    # Write to excel file
    with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
        retention_time_is_styled.to_excel(writer, sheet_name=sheet_name)
    # Use "publication" sample names in final table
    reindex_in_excel(filename=processed_filepath_xlsx, sheetname=sheet_name,)

# Method Detection Limits
The following block computes method detection limits (MDL) based on average and standard deviation ($\sigma$) of PFAS concentrations in selected samples (process blanks, etc.). \
The code reads in instrument detection limits (IDL) for the PFAS compounds. For some compounds the IDL may not be included in the input files, in that case the default value is used. \
Before the MDL calculation, **all concentrations below the IDL are replaced with the IDL** in order to ensure that MDLs are never lower than IDLs, and that the standard deviation is low in case of multiple concentrations detected below IDL.\
In case no concentrations could be detected at all, the IDL is used as MDL.

Within the Code the following definition of method detection limits is considered:\
$MDL_{PFAS} = average_{PFAS} + 3 \cdot \sigma_{PFAS}$

# Visual Sanity Check 2
The range of concentrations of selected blank samples for MDL calculations, IDL, and final MDL are depicted in the figure below. Outliers are shown and flagged. If the MDL for a certain PFAS compound is unreasonably high, you should not report/not trust the results for the corresponding compound and troubleshoot for contamination.

# Quality Control Criterion 2:
A new column 'Below Detection Limit (BDL)' is introduced, which indicates all 'Calculated Concentration' Values of PFAS quantification below the determined detection limits. These samples will be flagged as 'BDL' in the final concentration tables.

In [ ]:
### MDL calculation
# Initialize empty dataframe with IDL, relevant data for MDL calculation and MDL
# considering both channels MS/MS and TOF
na_list = [np.nan] * len(pfas_compounds_msms)
mdl = pd.DataFrame({
    'IDL': na_list, 'Mean Concentration MS/MS': na_list, 'Std Concentration MS/MS':na_list, 'MDL MS/MS': na_list,
    'Detection Limit MS/MS': na_list, f'Detection Limit MS/MS [ng/{sample_dominant_unit}]':na_list,
    'Mean Concentration TOF': na_list, 'Std Concentration TOF':na_list, 'MDL TOF': na_list,
    'Detection Limit TOF': na_list, f'Detection Limit TOF [ng/{sample_dominant_unit}]':na_list,
},index=pfas_compounds_msms)

# Load idl values from idl input file
idl = pd.read_csv(idl_filepath, index_col=0, low_memory=False, nrows=1)

# Write each idl value to idl column of MDL data frame and use default value idl_unknown if data is not available.
for row_index in mdl.index:
    if row_index in idl.columns:
        mdl.loc[row_index, 'IDL'] = idl[f'{row_index}'].to_list()[0]
    else:
        mdl.loc[row_index, 'IDL'] = idl_unknown
        print(f'No IDL available for {row_index}, default value of {idl_unknown} is used.')

# Calculate MDLs if samples for MDL calculations have been selected.
# Detects outliers and labels them
if not mdl_only is None:
    # select only target analytes and keep them in the right order
    blank_only_default = mdl_only[mdl_only['Component Name'].isin(pfas_compounds_msms + pfas_compounds_tof)]
    blank_only_default['Component Name'] = pd.Categorical(
         blank_only_default['Component Name'], categories=pfas_compounds_msms + pfas_compounds_tof, ordered=True
         )
    blank_only_default.sort_values('Component Name', inplace=True)
    # group data selected for MDL calculations (blanks) by PFAS compound
    mdl_groups = blank_only_default.groupby('Component Name', sort=False)

    # loops over groups (PFAS compounds)
    for (title, group) in mdl_groups['Calculated Concentration']:
        # get IDL from PFAS compound (title)
        # if statement for MS/MS channel, and TOF channel, respectively
        if not '_TOF MS' in title:
            relevant_idl = mdl.loc[title, 'IDL']
        else:
            relevant_idl = mdl.loc[title[:-7], 'IDL']
        # set all values below IDL to IDL
        group = group.apply(lambda x: x if x > relevant_idl else relevant_idl)

        # calculate mean and standard deviation of concentrations in blank and save it to mdl data frame
        # distinguish between MSMS channel and TOF channel in if statement
        if not '_TOF MS' in title:
            mdl.loc[title, 'Mean Concentration MS/MS'] = group.mean()
            mdl.loc[title, 'Std Concentration MS/MS'] = group.std()
        else:
            mdl.loc[title[:-7], 'Mean Concentration TOF'] = group.mean()
            mdl.loc[title[:-7], 'Std Concentration TOF'] = group.std()

    # calculate MDL
    mdl['MDL MS/MS'] = (mdl['Mean Concentration MS/MS'] + 3 * mdl['Std Concentration MS/MS']).apply(lambda x: round_to_n_sigfigs(x, 4))
    mdl['MDL TOF'] = (mdl['Mean Concentration TOF'] + 3 * mdl['Std Concentration TOF']).apply(lambda x: round_to_n_sigfigs(x, 4))

# use MDL as detection threshold -> use IDL if MDL is not available (NaN)
mdl[['Detection Limit MS/MS', 'Detection Limit TOF']] = mdl[['MDL MS/MS', 'MDL TOF']]
mdl[['Detection Limit MS/MS', 'Detection Limit TOF']].fillna(mdl.IDL, inplace=True)

# normalize detection limit to average sample size
mdl[[f'Detection Limit MS/MS [ng/{sample_dominant_unit}]', f'Detection Limit TOF [ng/{sample_dominant_unit}]']] = \
    mdl[['Detection Limit MS/MS', 'Detection Limit TOF']] / sample_average_size

# write detection limits to excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    mdl.to_excel(writer, sheet_name='Detection Limit')

In [ ]:
### MDL plots
# create plot to show details of MDL calculation
if not mdl_only is None:
    # loop over MS/MS and TOF compounds
    for (compounds, channel_type) in zip([pfas_compounds_msms, pfas_compounds_tof], ['MS/MS', 'TOF']):

        # creat image path and initialize figure
        image_path = os.path.join(plot_directory, f"MDL_{''.join(channel_type.split('/'))}.png")
        fig, ax = plt.subplots(figsize=(8, 8))

        idx = 1  # initialize index
        # loop over elements of groups
        # one group corresponds to one pfas compound with multiple values corresponding to multiple blank samples
        for (title, group) in mdl_groups:
            # skip if pfas compound is not in compound list
            if not title in compounds:
                continue
            # create boxplot for pfas compound
            ax.boxplot(group['Calculated Concentration'].dropna().values, positions=[idx], widths=0.5)
            
            # detect outliers for pfas compounds according to definition in boxplot
            outlier_threshold = group['Calculated Concentration'].quantile(0.75) + \
                1.5 * (group['Calculated Concentration'].quantile(0.75) - group['Calculated Concentration'].quantile(0.25))
            outliers = group.loc[group['Calculated Concentration'] > outlier_threshold, :]
            # loop over outliers and add labels in plot
            for (_, row) in outliers.iterrows():
                if row['Calculated Concentration'] < concentration_limit_plot:
                    ax.text(idx, row['Calculated Concentration'], row['Sample Name'], rotation=90, va='bottom', ha='right')
            idx +=1

        # plot mdl and idl
        ax.plot([np.nan] + (mdl['IDL']).to_list(), color='blue', linestyle='', marker="o",)
        ax.plot([np.nan] + (mdl[f'MDL {channel_type}']).to_list(), color='red', linestyle='', marker="*",)

        # axis, legend, descriptions, etc.
        ax.grid()
        ax.set_xticks(range(len(pfas_compounds_msms) + 1))
        ax.set_xticklabels([''] + pfas_compounds_msms, rotation=90)
        ax.set_ylim([0, concentration_limit_plot])
        fig.suptitle('')
        ax.set_title('')
        plt.ylabel('Concentration [ng/sample]')
        plt.xticks(rotation=90)
        box_patch = mpatches.Patch(color='black', fill=False, label='concentrations')
        blue_dot = Line2D([0], [0], marker='o', color='blue', label='IDL',)
        red_star = Line2D([0], [0], marker='*', color='red', label='MDL',)
        plt.legend(handles=[box_patch, blue_dot, red_star])
        plt.savefig(image_path, bbox_inches='tight')
        plt.show()

        # save mdl box plot to excel file
        workbook = load_workbook(processed_filepath_xlsx)
        plot_sheet = workbook.create_sheet(f"{''.join(channel_type.split('/'))} Detection Limits Plot")
        img = Image(image_path)
        cell_position = plot_sheet.cell(row=1, column=1).coordinate
        plot_sheet.add_image(img, cell_position)
        workbook.save(processed_filepath_xlsx)

In [ ]:
### Concentration values with MDL flags in excel
# initialize new concentration_data frame and select columns
selected_columns = [
    'Sample Name', 'Sample Index', 'Component Name', 'Calculated Concentration'
]

concentration_data = quantification_only[selected_columns]

# select only channel names of precursor masses (confirmation)
concentration_data = concentration_data[concentration_data['Component Name'].isin(compounds_msms + compounds_tof)]

# initialize new column BDL indicating the limits
concentration_data['BDL'] = False

for (index, row) in concentration_data.iterrows():
    # get detection limit from compound name
    compound_name = row['Component Name']
    # distinguish between TOF channel and MSMS channel
    if '_TOF MS' in compound_name:
        detection_limit = mdl.loc[compound_name[:-7], 'Detection Limit TOF']
    else:
        detection_limit = mdl.loc[compound_name, 'Detection Limit MS/MS']
    # check if concentration is below detection limit
    if abs(row['Calculated Concentration']) < detection_limit:
        concentration_data.loc[index, 'BDL'] = True  # flag True if above limit

concentration_data['Calculated Concentration'] = concentration_data['Calculated Concentration'].apply(lambda x: round_to_n_sigfigs(x, 4))
nan_flag = concentration_data['Calculated Concentration'].isnull()  # save mask of NaN concentration values
concentration_data.loc[nan_flag, 'BDL'] = np.nan  # set mask values of nan concentrtion values to np.nan

for (compounds, sheet_name) in zip(
    [pfas_compounds_msms, pfas_compounds_tof], ["MSMS Concentrations in ng MDL", "TOF Concentrations in ng MDL"]
    ):

    # Put concentration data in pivot tables
    concentration_data_pivot = concentration_data.pivot_table(
        index=('Sample Index',), columns='Component Name', values='Calculated Concentration', dropna=False,
    )[compounds]
    concentration_data_pivot_flag = concentration_data.pivot_table(
        index=('Sample Index',), columns='Component Name', values='BDL', dropna=False,
    )[compounds]

    # color cells based on threshold values
    concentration_data_pivot_styled = concentration_data_pivot.style.apply(
        highlight, flag=concentration_data_pivot_flag.fillna(value=False), axis=None
        )

    # Write to excel file
    with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
        concentration_data_pivot_styled.to_excel(writer, sheet_name=sheet_name)
    # Use "publication" sample names in final table
    reindex_in_excel(filename=processed_filepath_xlsx, sheetname=sheet_name,)

# Quality Control Criterion 3: Recovery Rates
To avoid misunderstandings, in the following two abbreviations are extensively used:
- IDA: isotope dilution analysis, also known as SS=surrogate standard or EIS=extracted internal standard
- IPS: isotope performance standard, also known as IS=injection standard or NIS=non-extracted internal standard

**The following two blocks calculate response factors from calibration data:**\
ratio of (i) calculated area of IDA * **actual concentration of IPS** and (ii) calculated area of IPS * actual concentration of IDA. \
The data is saved to an excel file and a boxplot of response factors is created.

Note: The response factor calculation within sciex uses the ratio of (i) calculated area of IDA and (ii) calculated area of IPS * actual concentration of IDA. \
As the concentration of IPS is missing in the calculation, the response factors deviate by a factor of 4, which is the actual concentration of IPS in the calibration data. \
Note: The column 'Actual Concentration' provides information on ng of internal standard added per sample.

**The third and forth code block calculate recovery rates for each IDA compound in each sample:**
$$
recovery~rate = \frac{\frac{area_{IDA~sample}~\cdot~concentration_{IPS~sample}}{area_{IPS~sample}~\cdot~concentration_{IDA~sample}}}{average(\frac{area_{IDA~calibration}~\cdot~concentration_{IPS~calibration}}{area_{IPS~calibration}~\cdot~concentration_{IDA~calibration}})} = \frac{IDA-IPS~ratio}{response~factor}
$$

In [ ]:
# define funtion which calculates the IDA IPS ratio.
# challenge - search right IPS row indicated in the Component Group Name of IDA.
def calculate_ida_ips_ratio(data: pd.DataFrame, column_name:str, ) -> pd.DataFrame:
    """Calculates IDA area times IPS concentration divided by IPS area times IDA concentration and save the results in the indicated column.

    :param data: Entire data junk (including all rows and the following columns:
    Component Name, Sample Index, Component Group Name, Actual Concentration, Area
    :type data: pd.DataFrame
    :param column_name: name of column, the calculated ratio should be saved to
    :type column_name: str
    :return: Data junk only containing IDA rows with the corresponding ratio saved to new column
    :rtype: pd.DataFrame
    """
    # select only ida rows from input data
    data_only_ida = data[data['Component Name'].str.contains(eis_identifier)]
    # initialize new column names
    data_only_ida[[f'{column_name}', 'IPS Area', 'IPS Concentration']] = np.nan

    # calculate recovery rate for every component, end every sample
    for row_index in data_only_ida.index:
        sample_index = data_only_ida.loc[row_index, 'Sample Index']
        ips_channel_name = data_only_ida.loc[row_index, 'Component Group Name']

        sample_name = data_only_ida.loc[row_index, 'Sample Name']
        corresponding_ips_area_row = data[(
            (data['Sample Index'] == sample_index) &
            (data['Component Name'] == ips_channel_name) &
            (data['Sample Name'] == sample_name)
            )]
        data_only_ida.loc[row_index, 'IPS Area'] = corresponding_ips_area_row['Area'].iloc[0]
        data_only_ida.loc[row_index, 'IPS Concentration'] = corresponding_ips_area_row['Actual Concentration'].iloc[0]
        data_only_ida.loc[row_index, f'{column_name}'] = \
            (data_only_ida.loc[row_index, 'Area'] * corresponding_ips_area_row['Actual Concentration'].iloc[0]) \
            / (corresponding_ips_area_row['Area'].iloc[0] * data_only_ida.loc[row_index, 'Actual Concentration'])
    return data_only_ida

In [ ]:
# Extract values of IDAs and IPS
# Save basic sample information as well as areas of intensity peak, actual concentration and Component Group Name.
# The 'Component Group Name' is useful to assoiciate the right IPS to each IDA.

# calucluate IDA IPS ratio to compute response factors with calibration data
calibration_only_ida = calculate_ida_ips_ratio(
    data=calibration_data, column_name='Response Factor Mean',
    )

# create data frame with this response factor calculation (from scratch), the standard deviation and the original values evaluated by Sciex,
response_factor = calibration_only_ida.groupby('Component Name', as_index=False)['Response Factor Mean'].mean()
response_factor['Response Factor Std'] = calibration_only_ida.groupby('Component Name')['Response Factor Mean'].std().to_list()
response_factor.index = response_factor['Component Name']
response_factor = response_factor.drop(columns=['Component Name']).reindex(eis_sorted)

# write response factor to excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    response_factor.to_excel(writer, sheet_name='Response Factor')

# create and save response factor box plots
image_path = os.path.join(plot_directory, 'response_factors.png')
fig, ax = plt.subplots(figsize=(8, 8))

response_factor_grouped = calibration_only_ida.groupby('Component Name', sort=False)
response_factor_grouped.boxplot(column='Response Factor Mean', ax=ax, subplots=False)

# calibration_only_ida.boxplot(column='Response Factor Mean', by='Component Name', ax=ax,)
ax.set_xticks(range(len(response_factor_grouped) + 1))
ax.set_xticklabels([''] + eis_sorted, rotation=90)
fig.suptitle('')
ax.set_title('')
plt.ylabel('Response Factor (IDA area/IPS area)')
plt.xticks(rotation=90)
plt.savefig(image_path, bbox_inches='tight')
plt.show()

# save response factor box plot to excel file
workbook = load_workbook(processed_filepath_xlsx)
plot_sheet = workbook['Response Factor']
img = Image(os.path.join(plot_directory, 'response_factors.png'))
img.anchor = 'F1'
plot_sheet.column_dimensions['F'].width = img.width / 6
plot_sheet.add_image(img)
workbook.save(processed_filepath_xlsx)

Recovery rate computed within the SCIEX software is not normalized by concentration. So the recovery rate calculated by SCIEX reads:
$$
recovery~rate = \frac{\frac{area_{IDA~sample}}{area_{IPS~sample}}}{average(\frac{area_{IDA~calibration}}{area_{IPS~calibration}})}
$$

The uncertainty of the recovery rate is indicated by a maximum error method - for now, you can just ignore this part...
$$
\Delta recovery~rate = \Delta response~factor \cdot \frac{ratio}{response~factor^{2}} + \Delta ratio \cdot \frac{1}{response~factor}
$$

In [ ]:
# function to determine if recovery rates are within given limit
def flag_poor_recovery(data: pd.DataFrame, checked_column_name: str, new_column_name: str, columns_sorted: list[str]) -> tuple[pd.DataFrame, None]:
    """Checks if recovery rates are in defined threshold limits and appends check results as boolean column to data.
    Additionally formats table for recovery rate.

    :param data: Entire data junk including all rows and the following columns:
    'Component Name', 'Sample Index', f'{checked_column_name}'
    :type data: pd.DataFrame
    :param checked_column_name: name of column which is used to check if values are in the given limits.
    :type checked_column_name: str
    :param new_column_name: name of column, the check results should be saved to.
    :type new_column_name: str
    :param columns_sorted: List indicating the order of columns of pivot tables
    :type columns_sorted: list[str]
    :return: Original data junk with the new column (ckeck results) appended + formated pivot table containing recovery rates.
    :rtype: tuple[pd.DataFrame, pd.DataFrame.style.Styler]
    """
    data[f'{new_column_name}'] = np.nan
    for component_ida in data['Component Name'].unique():
        lower_threshold_recovery = recovery_thresholds_dict['lower threshold for recoveries [%]'][component_ida]
        upper_threshold_recovery = recovery_thresholds_dict['upper threshold for recoveries [%]'][component_ida]
        data.loc[
            (data['Component Name'] == component_ida) & 
            (~data[f'{checked_column_name}'].isnull())
        , f'{new_column_name}'] = True
        data.loc[
            (data['Component Name'] == component_ida) &
            (data[f'{checked_column_name}'] > lower_threshold_recovery) &
            (data[f'{checked_column_name}'] < upper_threshold_recovery)
        , f'{new_column_name}'] = False

    recovery_pivot = data.pivot_table(
        index=('Sample Index',), columns='Component Name', values=f'{checked_column_name}', dropna=False,
    )[columns_sorted]

    poor_recovery_pivot = data.pivot_table(
        index=('Sample Index',), columns='Component Name', values=f'{new_column_name}', dropna=False,
        )[columns_sorted]

    # color cells based on recovery values
    recovery_styled = recovery_pivot.style.apply(highlight, flag=poor_recovery_pivot.fillna(value=True), axis=None)

    return data, recovery_styled


In [ ]:
# Select ida rows from quantification data and calculate ida ips ratio
# function is defined in previous block
quantification_ida = calculate_ida_ips_ratio(
    data=sample_data, column_name="IDA-IPS Ratio",
    )

recovery_table = pd.DataFrame(columns=[
    'Sample Index', 'Component Name', 'IDA-IPS Ratio',
    'Response Factor Mean', 'Recovery Rate',
    ])

# delete IDA and IPS values from one method if 'core' or 'extended' is selected,
# and use the average of both methods in the alternative case.
index = 0

for sample_index in quantification_ida['Sample Index'].unique():
    for component_ida in quantification_ida['Component Name'].unique():
        selected_sample_ida = quantification_ida.loc[(
            (quantification_ida['Sample Index'] == sample_index) & 
            (quantification_ida['Component Name'] == component_ida)
        ), :]
        response_factor_mean = response_factor.loc[component_ida, 'Response Factor Mean']
        if len(selected_sample_ida.index) == 0:
            print(f'Component {component_ida} is not available for sample {sample_index}')
            ida_ips_ratio_mean = np.nan
        else:
            ida_ips_ratio_mean = selected_sample_ida['IDA-IPS Ratio'].values[0]

        recovery_rate = 100 * ida_ips_ratio_mean / response_factor_mean
        recovery_table.loc[index] = [
            sample_index, component_ida, ida_ips_ratio_mean,
            response_factor_mean, recovery_rate,
            ]
        index +=1

# check if recovery rate is within indicated limit for each ida and save check results to 'Poor Recovery column
recovery_table, recovery_styled = flag_poor_recovery(data=recovery_table, checked_column_name='Recovery Rate', new_column_name='Poor Recovery', columns_sorted=eis_sorted)

# Write recovery rate to excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    recovery_thresholds.to_excel(writer, sheet_name='Recovery Thresholds')
    recovery_styled.to_excel(writer, sheet_name='MSMS Recovery Rate')
    
reindex_in_excel(filename=processed_filepath_xlsx, sheetname='MSMS Recovery Rate')

The following code cells generate plots of recovery rates over all samples.

In [ ]:
# Box plot for recovery rates
image_path = os.path.join(plot_directory, 'recovery_rates_box.png')
fig, ax = plt.subplots(figsize=(8, 8))
recovery_grouped = recovery_table.groupby('Component Name', sort=False)
recovery_grouped.boxplot(column='Recovery Rate', ax=ax, subplots=False)
ax.set_ylim([0, recovery_limit_plot])
ax.set_xticks(range(len(recovery_grouped) + 1))
ax.set_xticklabels([''] + eis_sorted, rotation=90)
fig.suptitle('')
ax.set_title('')
plt.ylabel('Recovery Rate')
plt.xticks(rotation=90)
plt.legend()
plt.savefig(image_path, bbox_inches='tight')
plt.show()

# plot data as points for recovery rates:
plot_data = recovery_table.groupby('Sample Index')  # group data for plotting
cmap = plt.cm.get_cmap('tab20', len(plot_data)) # initialize colours
image_path = os.path.join(plot_directory, 'recovery_rates.png')  # set path for figure

fig, ax = plt.subplots(figsize=(8, 8))
for index, (title, group) in enumerate(plot_data):
    group.set_index(group['Component Name'], inplace=True)
    group.drop_duplicates(keep='first', inplace=True)
    group.plot(
        y='Recovery Rate', ax=ax, marker='.', linestyle='None', label=sample_number_mapper_sample_name[title],
        grid=True, color = cmap(index),
    )
ax.set_ylim([0, recovery_limit_plot])
fig.suptitle('')
ax.set_xticks(range(len(group)))
ax.set_xticklabels(group['Component Name'], rotation=90)
ax.set_title('')
plt.xticks(rotation=90)
plt.ylabel('Recovery Rate')
plt.legend(loc='center right', bbox_to_anchor=(1.4, 0.5))
plt.savefig(image_path, bbox_inches='tight')
plt.show()

# save recovery rate plot to excel file
workbook = load_workbook(processed_filepath_xlsx)
plot_sheet = workbook.create_sheet('MSMS Recovery Rates Plots')

img1 = Image(os.path.join(plot_directory, 'recovery_rates_box.png'))
img1.anchor = 'A1'
plot_sheet.column_dimensions['A'].width = img1.width / 6
plot_sheet.row_dimensions[1].height = img.height
plot_sheet.add_image(img1)

img2 = Image(os.path.join(plot_directory, 'recovery_rates.png'))
img2.anchor = 'B1'
plot_sheet.column_dimensions['B'].width = img2.width / 6
plot_sheet.add_image(img2)

workbook.save(processed_filepath_xlsx)

# Quality Criterion 4: Ion abundance ratios
The following block evaluates ion abundance ratios from midpoint calibration data, computes the average and plots the results. \
The ion abundance ratio (IAR) is caluclated for each PFAS compound and each standard as follows:

$IAR_{PFAS} = \frac{Area-TOF_{PFAS}}{Area-MSMS_{PFAS}}$

In [ ]:
# Assign TOF channel to each MS MS channel and caluclate ion abundance ratio. 
# define funtion which calculates the ion abundance ratio
# challenge - search right TOF corresponding to MS MS.
def calculate_ion_abundance_ratio(pfas_components_tof: list[str], pfas_components_msms: list[str], data:pd.DataFrame) -> pd.DataFrame:
    """Calculate ion abundance ratio of a given data set.

    :param pfas_components_tof: List of all PFAS compounds having a TOF_MS channel in the right order.
    :type pfas_components_tof: list[str]
    :param pfas_components_msms: List of all PFAS compounds having an MSMS channel in the right order.
    :type pfas_components_msms: list[str]
    :param data:  Entire data junk (including all rows and the following columns:
    Component Name, Sample Index, Sample Name, Area
    :type data: pd.DataFrame
    :return:  Data junk containing area of tof component and ion abundance ratio for msms channels as new column.
    :rtype: pd.DataFrame
    """
    # Initialize new columns and set them to NaN per default
    data['Ion Abundance Ratio'] = np.nan
    data['TOF Area'] = np.nan

    # get index of msms component of exact same sample and tof component of exact same sample
    for sample_index in data['Sample Index'].unique():
        for (msms_component, tof_component) in zip(pfas_components_msms, pfas_components_tof):
            msms_row_index = data[(
                (data["Sample Index"] == sample_index) &
                (data['Component Name'] == msms_component))].index.to_list()
            tof_row_index = data[(
                (data["Sample Index"] == sample_index) &
                (data['Component Name'] == tof_component))].index.to_list()

            # find right msms channel row in case two components are availeble (core vs. extended method)
            if len(msms_row_index) == 0:
                print(f'Component {msms_component} is not available for sample {sample_number_mapper_sample_name[sample_index]}.')
                msms_area = np.nan
            else:
                msms_area = data.loc[msms_row_index[0], 'Area']

            # find right tof channel row in case two components are availeble (core vs. extended method)
            if len(tof_row_index) == 0:
                print(f'Component {tof_component} is not available for sample {sample_number_mapper_sample_name[sample_index]}.')
                tof_area = np.nan
            else:
                tof_area = data.loc[tof_row_index[0], 'Area']
    
            # calculate IAR if both msms channel and tof ms channel are available
            if not (np.isnan(msms_area) or np.isnan(tof_area)):
                data.loc[msms_row_index, 'TOF Area'] = tof_area
                data.loc[msms_row_index, 'Ion Abundance Ratio'] = tof_area / msms_area
    return data

In [ ]:
# select relavant columns of calibration midpoint data
selected_columns = [
    'Sample Index', 'Sample Name', 'Component Name', 'Area',
    ]
mass_channel_area_calibration_midpoint = calibration_midpoint_data[selected_columns]

# calculate ion abundance ratios and drop tof channel rows afterwards
ion_abundance_ratio_calibration_midpoint = calculate_ion_abundance_ratio(
    pfas_components_msms=pfas_compounds_msms + eis_nis_msms, pfas_components_tof=pfas_compounds_tof + eis_nis_tof,
    data=mass_channel_area_calibration_midpoint
)
ion_abundance_ratio_calibration_midpoint = ion_abundance_ratio_calibration_midpoint[
    ~ion_abundance_ratio_calibration_midpoint['Component Name'].str.contains('_TOF MS')
]

# group ion abundance ratios in calibration midpoint by component
ion_abundance_ratio_calibration_midpoint_grouped = ion_abundance_ratio_calibration_midpoint.groupby(
    'Component Name', as_index=True, sort=False)

# evaluate mean of groups - ion abundance ratio per compound in calibration midpoint
ion_abundance_ratio_calibration_midpoint_mean = ion_abundance_ratio_calibration_midpoint_grouped['Ion Abundance Ratio'].mean()

# write ion abundance ratio in calibration midpoint to excel
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    ion_abundance_ratio_calibration_midpoint_mean.to_excel(
        writer, sheet_name=f'{calibration_midpoint_identifier} IAR'
        )

# create and save response factor box plots
image_path = os.path.join(plot_directory, 'midpoint_ion_abundance_ratio.png')
fig, ax = plt.subplots(figsize=(16, 8))
ion_abundance_ratio_calibration_midpoint_grouped.boxplot(column='Ion Abundance Ratio', ax=ax, subplots=False)

# calibration_only_ida.boxplot(column='Response Factor Mean', by='Component Name', ax=ax,)
ax.set_xticks(range(len(ion_abundance_ratio_calibration_midpoint_grouped ) + 1))
ax.set_xticklabels([''] + pfas_compounds_msms + eis_nis_msms, rotation=90)
fig.suptitle('')
ax.set_title('')
plt.ylabel('Ion Abundance Ratio(TOF channel area / MS-MS channel area)')
plt.xticks(rotation=90)
plt.savefig(image_path, bbox_inches='tight')
plt.show()

# save image to excel
workbook = load_workbook(processed_filepath_xlsx)
plot_sheet = workbook[f'{calibration_midpoint_identifier} IAR']
img = Image(image_path)
img.anchor = 'D1'
plot_sheet.column_dimensions['D'].width = img.width / 6
plot_sheet.add_image(img)
workbook.save(processed_filepath_xlsx)

The following code block evaluates ion abundance ratios in the quantification data and compares them to the ion abundance ratios found in the calibration midpoint (CSM) to evaluate the percentage deviation:

$IAR~deviation_{sample, PFAS} = 100 \cdot (\frac{IAR_{sample, PFAS}}{mean(IAR_{CSM,PFAS})} - 1)$

In [ ]:
# initialize new ion abundance ratio data frame and select columns
ion_abundance_ratio = sample_data[selected_columns]

# calculate ion abundance ratio for all compounds and standards and delete rows from TOF channel afterwards
ion_abundance_ratio = calculate_ion_abundance_ratio(
    pfas_components_msms=pfas_compounds_msms + eis_nis_msms, pfas_components_tof=pfas_compounds_tof + eis_nis_tof,
    data=ion_abundance_ratio
)
ion_abundance_ratio = ion_abundance_ratio[
    ~ion_abundance_ratio['Component Name'].str.contains('_TOF MS')
]

# Initialize new Ion Abundance Ratio Deviation column and set to NaN per default.
ion_abundance_ratio['Ion Abundance Ratio Deviation [%]'] = np.nan

for sample_index in ion_abundance_ratio['Sample Index'].unique():
    for compound in ion_abundance_ratio['Component Name'].unique():
        selected_index = ion_abundance_ratio[(
            (ion_abundance_ratio['Sample Index'] == sample_index) & 
            (ion_abundance_ratio['Component Name'] == compound)
        )].index.to_list()
        if len(selected_index) == 1:
            ion_abundance_ratio_calibration_midpoint = ion_abundance_ratio_calibration_midpoint_mean.loc[compound]
            ion_abundance_ratio.loc[selected_index[0], 'Ion Abundance Ratio Deviation [%]'] = \
                100 * (ion_abundance_ratio.loc[selected_index[0], 'Ion Abundance Ratio'] / ion_abundance_ratio_calibration_midpoint - 1)
        else:
            print(sample_index, sample_number_mapper_sample_name[sample_index])
# Introduce new column where everything above or below 50 % deviation is flagged
nan_flag = ion_abundance_ratio['Ion Abundance Ratio Deviation [%]'].isnull()
ion_abundance_ratio[f'Ion Abundance Ratio Deviation > {iard_threshold}'] = abs(ion_abundance_ratio['Ion Abundance Ratio Deviation [%]']) > iard_threshold
ion_abundance_ratio.loc[nan_flag, f'Ion Abundance Ratio Deviation > {iard_threshold}'] = np.nan

# Put channel ratio in pivot table
ion_abundance_ratio_table = ion_abundance_ratio.pivot_table(
    index=('Sample Index',), columns='Component Name', values='Ion Abundance Ratio Deviation [%]', dropna=False,
)[pfas_compounds_msms + eis_nis_msms]
ion_abundance_ratio_flag = ion_abundance_ratio.pivot_table(
    index=('Sample Index',), columns='Component Name', values=f'Ion Abundance Ratio Deviation > {iard_threshold}', dropna=False,
)[pfas_compounds_msms + eis_nis_msms]

# color cells based on threshold values
ion_abundance_ratio_styled = ion_abundance_ratio_table.style.apply(
    highlight, flag=ion_abundance_ratio_flag.fillna(value=False), axis=None
    )

# Write to excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    ion_abundance_ratio_styled.to_excel(writer, sheet_name='IAR Deviation in %')
reindex_in_excel(filename=processed_filepath_xlsx, sheetname='IAR Deviation in %')

# Ouputs and data merging

In the following code block all relevant data and parameters are combined in one common dataframe quantification_pfas_default, which has one row for each (MSMS) PFAS compound in each sample: \
(1) The recovery rate is assigned to each (MSMS) PFAS compound. Each PFAS compound has a corresponding IDA - the recovery rate is deduced from the recovery rate of the corresponding IDA. The assignment works for each sample by using the 'IS Name' column which provides information on which IDA standard is associated to which PFAS. In adddition, a boolean column is appended, indicating if the recovery rates are in the allowed limits.\
(2) detection limits are appended to the table as well as a boolean column indicating if the detected values are below the threshold. \
(3) basic information about the sample are included in the final data frame. \
(4) ion abundance ratios deviations are included, as well as a boolean column inidcating if the QAQC criteria are met.
(5) retention time differences are included in the final table with respect to internal standards and the same compounds in the calibration midpoint. For both options a boolean column inidcating if the QAQC criteria are met is appended.

In [ ]:
# Initialize data frame for following assignments, select only needed columns and use reasonable naming.
selected_columns = [
    'Sample Name', 'Sample Index', 'Acquisition Date & Time', 'Component Name', 'Calculated Concentration',
    'Area', 'IS Name',
    ]
quantification_pfas_default = quantification_only[selected_columns]
quantification_pfas_default.rename(columns={'IS Name': 'IDA Name'}, inplace=True)

# Get only PFAS default channels
quantification_pfas_default = quantification_pfas_default[
    quantification_pfas_default['Component Name'].isin(pfas_compounds_msms)
    ]

# initialize new columns
quantification_pfas_default[[
    'RTD', 'Sample Quantity', 'Sample Unit', 'BDL', 'Recovery Rate', 'Poor Recovery',
    'IAR Deviation', 'IARD', 'flag',
]] = np.nan

# , 'IDA Area', 'IDA Concentration', 'IPS Name', 'IPS Area', 'IPS Concentration', 'TOF Area', 

for row_index in quantification_pfas_default.index:
    sample_index = quantification_pfas_default.loc[row_index, 'Sample Index']
    sample_name = quantification_pfas_default.loc[row_index, 'Sample Name']
    ida_channel_name = quantification_pfas_default.loc[row_index, 'IDA Name']
    component_name = quantification_pfas_default.loc[row_index, 'Component Name']

    rt_row = retention_time_data[(
        (retention_time_data['Component Name'] == component_name) &
        (retention_time_data['Sample Index'] == sample_index)
    )]
    
    bdl_row = concentration_data[(
        (concentration_data['Component Name'] == component_name) &
        (concentration_data['Sample Index'] == sample_index)
    )]

    ida_row = quantification_ida[(
            (quantification_ida['Component Name'] == ida_channel_name) &
            (quantification_ida['Sample Index'] == sample_index) &
            (quantification_ida['Sample Name'] == sample_name)
    )]

    recovery_row = recovery_table[(
        (recovery_table['Component Name'] == ida_channel_name) &
        (recovery_table['Sample Index'] == sample_index)
    )]

    iar_row = ion_abundance_ratio[(
            (ion_abundance_ratio['Component Name'] == component_name) &
            (ion_abundance_ratio['Sample Index'] == sample_index)
    )]

    # assign Sample Quantity and Sample Unit
    quantification_pfas_default.loc[row_index, 'Sample Quantity'] = \
        sample_input_data.loc[sample_index, 'volume/weight/number of samples']
    quantification_pfas_default.loc[row_index, 'Sample Unit'] = \
        sample_input_data.loc[sample_index, 'unit (e.g. g/mL/sample)']

    # assign RTD
    quantification_pfas_default.loc[row_index, 'RTD'] = rt_row.loc[:,'RTD'].values[0]

    # assign BDL
    quantification_pfas_default.loc[row_index, 'BDL'] = bdl_row.loc[:,'BDL'].values[0]

    # assign IDA Area and Concentration, IPS Name, Area and Concentration, as well as Recovery Rates
    if ida_channel_name in quantification_ida['Component Name'].to_list():
        # quantification_pfas_default.loc[row_index,'IDA Area'] = ida_row.loc[:,'Area'].values[0]
        # quantification_pfas_default.loc[row_index,'IDA Concentration'] = ida_row.loc[:,'Actual Concentration'].values[0]
        # quantification_pfas_default.loc[row_index,'IPS Area'] = ida_row.loc[:,'IPS Area'].values[0]
        # quantification_pfas_default.loc[row_index,'IPS Concentration'] = ida_row.loc[:,'IPS Concentration'].values[0]
        # quantification_pfas_default.loc[row_index, 'IPS Name'] = \
        #     ida_row.loc[:,'Component Group Name'].values[0]
        quantification_pfas_default.loc[row_index, 'Recovery Rate'] = \
            recovery_row.loc[:, 'Recovery Rate'].values[0]
        quantification_pfas_default.loc[row_index, 'RR'] = \
            recovery_row.loc[:, 'Poor Recovery'].values[0]

    # quantification_pfas_default.loc[row_index, 'TOF Area'] = \
    #     iar_row.loc[:, 'TOF Area'].values[0]
    quantification_pfas_default.loc[row_index, 'IAR Deviation'] = \
        iar_row.loc[:, 'Ion Abundance Ratio Deviation [%]'].values[0]
    quantification_pfas_default.loc[row_index, 'IARD'] = \
        iar_row.loc[:, f'Ion Abundance Ratio Deviation > {iard_threshold}'].values[0]

quantification_pfas_default['Recovery Rate'] = quantification_pfas_default['Recovery Rate'].apply(lambda x: round_to_n_sigfigs(x, 4))

# set flags accordingly
quantification_pfas_default.loc[quantification_pfas_default['IARD'].fillna(False), 'flag'] = 'IARD'
quantification_pfas_default.loc[quantification_pfas_default['RR'].fillna(False), 'flag'] = 'RR'
quantification_pfas_default.loc[quantification_pfas_default['BDL'].fillna(False), 'flag'] = 'BDL'
quantification_pfas_default.loc[quantification_pfas_default['RTD'].fillna(False), 'flag'] = 'RTD'

# Outputs
The following code block writes final concentration table with all information to excel:
- It uses retention time difference between compound and related internal standard and indicates all values above the allowed difference with 'RTD'\
- It uses concentration values and indicates all values below detection limit with 'BDL' \
- It uses recovery rates and indicates all values below or above the indicated thresholds with 'RR'. \
- It uses ion abundance ratio deviations and indicates all values below the allowed deviation with 'IARD' \

In addition, concentration values are converted to ng\g or ng\l respectively.

Writes final long format table to csv

In [ ]:
def flag_values_string(data: pd.DataFrame, column: str) -> pd.DataFrame:
    """Introduces new column to dataframe with concentrations as string and flag label when flagged.
    RTD...retention time difference
    BDL...below detection limit
    RR...recovery rate
    IARD...ion abundance ratio deviation

    :param data: data frame containing column 'flag',
    as well as the column you indicated.
    :type data: pd.DataFrame
    :param column: Colun name of data frame to be filtered or flagged.
    :type column: str
    :return: Data frame, where the column data is flagged.
    :rtype: pd.DataFrame
    """
    final_table = data[['Sample Name', 'Sample Index', 'Component Name', column]]
    final_table[column] = final_table[column].apply(lambda x: round_to_n_sigfigs(x, 4))
    for (row_index, row) in data.iterrows():
        if np.isnan(row[column]):
            row_conc = 'ND'
        else:
            row_conc = str(row[column])
            if not pd.isnull(row['flag']):
                row_conc = row['flag']
            else:
                row_conc = str(row[column])
        final_table.loc[row_index, column] = row_conc

    return final_table

def flag_values_na(data: pd.DataFrame, column: str) -> pd.DataFrame:
    """Flags column values (most probably concentrations) which do not meet QAQC criteria,
    and replaces them with NaNs.

    :param data: data frame containing column 'flag',
    as well as the column you indicated.
    :type data: pd.DataFrame
    :param column: Column name of data frame to be filtered or flagged.
    :type column: str
    :return: Data frame, where the column data is flagged.
    :rtype: pd.DataFrame
    """
    final_table = data[['Sample Name', 'Sample Index', 'Component Name', column]]
    final_table[column] = final_table[column].apply(lambda x: round_to_n_sigfigs(x, 4))
    for (row_index, row) in data.iterrows():
        if not pd.isnull(row['flag']):
            final_table.loc[row_index, column] = np.nan
    return final_table

# Transform concentration to ng/g or ng/l, depending on your sample_unit
quantification_pfas_default[f'Concentration in ng per Unit'] = quantification_pfas_default['Calculated Concentration'] \
    / quantification_pfas_default['Sample Quantity']

# Flag concentration values with channel ratio, recovery rates and detection threshold.
calculated_concentration_string = flag_values_string(data=quantification_pfas_default, column=f'Concentration in ng per Unit')
calculated_concentration_nan = flag_values_na(data=quantification_pfas_default, column=f'Concentration in ng per Unit')

# Pivot concentration tables.
calculated_concentration_string = calculated_concentration_string.pivot(
    index=('Sample Index',), columns='Component Name', values=f'Concentration in ng per Unit',
)
calculated_concentration_string = calculated_concentration_string[pfas_compounds_msms]

calculated_concentration_nan = calculated_concentration_nan.pivot_table(
    index=('Sample Index',), columns='Component Name', values=f'Concentration in ng per Unit', dropna=False,
)
calculated_concentration_nan = calculated_concentration_nan[pfas_compounds_msms]

# add unit to index of final concentration table
tuple_index = []
for sample_index in calculated_concentration_string.index:
    tuple_index.append((sample_number_mapper_sample_name[sample_index], 'ng/' + sample_number_mapper_sample_unit[sample_index]))

calculated_concentration_string.index = pd.Index(tuple_index)
calculated_concentration_nan.index = pd.Index(tuple_index)

# Write pivot tables to existing excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    calculated_concentration_string.to_excel(writer, sheet_name=f'Concentration (ng per Unit)')
    calculated_concentration_nan.to_excel(writer, sheet_name=f'Numeric Conc. (ng per Unit)')

# Write long format data to csv
quantification_pfas_default = quantification_pfas_default[[
    'Sample Name', 'Sample Index', 'Acquisition Date & Time','Component Name',
    'Calculated Concentration', 'Concentration in ng per Unit', 'Sample Unit',
    'flag',
]]
quantification_pfas_default['Component Name'] = pd.Categorical(quantification_pfas_default['Component Name'], categories=pfas_compounds_msms, ordered=True)
quantification_pfas_default.rename(columns={'Calculated Concentration': 'Mass in ng'}, inplace=True)
quantification_pfas_default.sort_values(by=['Sample Index', 'Component Name'], inplace=True)
quantification_pfas_default.index = [i for i in range(len(quantification_pfas_default))]

# rename samples to publication names
quantification_pfas_default['Sample Name'] = [sample_number_mapper_sample_name[sample_index] for sample_index in quantification_pfas_default['Sample Index'].to_list()]

quantification_pfas_default.to_csv(processed_filepath_csv)

# Print output
display(calculated_concentration_string)